In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Load dataset
df = pd.read_csv("../data/raw/PS_20174392719_1491204439457_log.csv")

# Recreate engineered features
df["orig_balance_change"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["dest_balance_change"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["orig_amount_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
).abs()
df["dest_amount_error"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
).abs()

features = [
    "step", "type", "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest", "orig_balance_change",
    "dest_balance_change", "orig_amount_error", "dest_amount_error"
]

# Take a stratified 200,000-row sample to keep this CPU-friendly
sample_df, _ = train_test_split(
    df,
    train_size=200_000,
    random_state=42,
    stratify=df["isFraud"]
)

X_sample = sample_df[features]
y_sample = sample_df["isFraud"].to_numpy()

# Encode transaction type and scale features
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["type"]),
    ("num", "passthrough", [c for c in features if c != "type"])
])

X_processed = preprocessor.fit_transform(X_sample)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed).astype(np.float32)

# Split into anomaly-model training and evaluation samples
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_scaled,
    y_sample,
    test_size=0.25,
    random_state=42,
    stratify=y_sample
)

print("Training shape:", X_train_np.shape)
print("Validation shape:", X_val_np.shape)
print("Validation fraud cases:", int(y_val_np.sum()))

Training shape: (150000, 15)
Validation shape: (50000, 15)
Validation fraud cases: 64


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score
)

isolation_model = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=0.0013,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train_np)

anomaly_predictions = (
    isolation_model.predict(X_val_np) == -1
).astype(int)

anomaly_scores = -isolation_model.score_samples(X_val_np)

print("Precision:", round(
    precision_score(y_val_np, anomaly_predictions, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val_np, anomaly_predictions, zero_division=0), 4
))
print("F1 Score:", round(
    f1_score(y_val_np, anomaly_predictions, zero_division=0), 4
))
print("PR-AUC:", round(
    average_precision_score(y_val_np, anomaly_scores), 4
))
print("Confusion Matrix:")
print(confusion_matrix(y_val_np, anomaly_predictions))

Precision: 0.0145
Recall: 0.0156
F1 Score: 0.015
PR-AUC: 0.0109
Confusion Matrix:
[[49868    68]
 [   63     1]]
